# ============================================================
# CELL 1 — XGBoost with scale_pos_weight + Best Params from RandomizedSearchCV
# Run this AFTER your existing Cell (Approach 4 / random_search)
# ============================================================

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# ── Step 1: Calculate scale_pos_weight from training labels ──
# This tells XGBoost to pay more attention to the minority class (Diabetes=1)
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f"Negative class count: {neg}")
print(f"Positive class count: {pos}")
print(f"scale_pos_weight: {spw:.2f}")

# ── Step 2: Grab the best params found by your RandomizedSearchCV ──
best_gb_params = random_search.best_params_
print(f"\nBest params from RandomizedSearchCV: {best_gb_params}")

# ── Step 3: Map GB params → XGBoost equivalents ──
# GradientBoostingClassifier and XGBClassifier use different param names
xgb_params = {
    'n_estimators':      best_gb_params.get('n_estimators', 200),
    'max_depth':         best_gb_params.get('max_depth', 4),
    'learning_rate':     best_gb_params.get('learning_rate', 0.1),
    'subsample':         best_gb_params.get('subsample', 0.8),
    'min_child_weight':  best_gb_params.get('min_samples_leaf', 1),  # closest XGB equivalent
}
print(f"\nMapped XGBoost params: {xgb_params}")

# ── Step 4: Train XGBoost ──
xgb_model = XGBClassifier(
    **xgb_params,
    scale_pos_weight=spw,      # handles class imbalance
    eval_metric='logloss',
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train_scaled, y_train)

# ── Step 5: Evaluate at default threshold (0.5) ──
y_prob_xgb  = xgb_model.predict_proba(X_test_scaled)[:, 1]
y_pred_xgb  = xgb_model.predict(X_test_scaled)

print("\n" + "="*55)
print("XGBoost — Default Threshold (0.5)")
print("="*55)
print(classification_report(y_test, y_pred_xgb))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob_xgb):.4f}")

# ============================================================
# CELL 2 — Cross-Validated Threshold Tuning
# Find the best threshold on TRAINING DATA, then apply to TEST
# ============================================================

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt

# ── Step 1: Generate out-of-fold (OOF) predicted probabilities on TRAINING data ──
# This is the key: we never touch X_test_scaled here
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros(len(y_train))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_scaled, y_train)):
    X_tr  = X_train_scaled[train_idx]
    X_val = X_train_scaled[val_idx]
    y_tr  = y_train.iloc[train_idx] if hasattr(y_train, 'iloc') else y_train[train_idx]
    y_val = y_train.iloc[val_idx]   if hasattr(y_train, 'iloc') else y_train[val_idx]

    # Retrain XGBoost on each fold's training split
    fold_model = XGBClassifier(
        **xgb_params,
        scale_pos_weight=spw,
        eval_metric='logloss',
        random_state=42,
        verbosity=0
    )
    fold_model.fit(X_tr, y_tr)
    oof_probs[val_idx] = fold_model.predict_proba(X_val)[:, 1]
    print(f"  Fold {fold+1} done")

print("\nOOF probabilities generated on all training samples.")

# ── Step 2: Sweep thresholds on OOF predictions ──
thresholds = np.arange(0.20, 0.65, 0.01)
f1_scores  = []

y_train_arr = y_train.values if hasattr(y_train, 'values') else y_train

for thresh in thresholds:
    preds = (oof_probs >= thresh).astype(int)
    f1_scores.append(f1_score(y_train_arr, preds))

# ── Step 3: Pick best threshold ──
best_thresh = thresholds[np.argmax(f1_scores)]
best_f1_cv  = max(f1_scores)
print(f"\nBest threshold (from CV): {best_thresh:.2f}")
print(f"Best CV F1 at that threshold: {best_f1_cv:.4f}")

# ── Step 4: Plot threshold vs F1 ──
plt.figure(figsize=(9, 4))
plt.plot(thresholds, f1_scores, color='steelblue', linewidth=2)
plt.axvline(x=best_thresh, color='tomato', linestyle='--', linewidth=1.5,
            label=f'Best threshold = {best_thresh:.2f}  (F1 = {best_f1_cv:.3f})')
plt.xlabel('Threshold')
plt.ylabel('Cross-Validated F1 Score')
plt.title('Threshold Tuning via Cross-Validation (on Training Data Only)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# CELL 3 — Final Evaluation with Best Threshold
# Apply the CV-chosen threshold to the test set (only here)
# ============================================================

In [ ]:
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, average_precision_score, precision_recall_curve
)

# Apply the threshold found via CV
y_pred_final = (y_prob_xgb >= best_thresh).astype(int)

print("="*55)
print(f"XGBoost — CV-Tuned Threshold ({best_thresh:.2f})")
print("="*55)
print(classification_report(y_test, y_pred_final))
print(f"AUC-ROC : {roc_auc_score(y_test, y_prob_xgb):.4f}")
print(f"Avg Prec: {average_precision_score(y_test, y_prob_xgb):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: Confusion Matrix ──
cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix (threshold={best_thresh:.2f})')

# ── Plot 2: ROC Curve (step function — correct shape) ──
fpr, tpr, _ = roc_curve(y_test, y_prob_xgb)
auc_roc = roc_auc_score(y_test, y_prob_xgb)
axes[1].plot(fpr, tpr, drawstyle='steps-post',     # <-- staircase, not smooth
             color='steelblue', linewidth=2,
             label=f'XGBoost (AUC = {auc_roc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (step-function)')
axes[1].legend()
axes[1].grid(alpha=0.3)

# ── Plot 3: Probability Distribution (KDE — smooth & readable) ──
axes[2].set_title('Predicted Probability Distribution by Class')
sns.kdeplot(y_prob_xgb[y_test == 0], fill=True, alpha=0.45,
            color='steelblue', label='No Diabetes (0)',
            bw_adjust=0.8, ax=axes[2])
sns.kdeplot(y_prob_xgb[y_test == 1], fill=True, alpha=0.45,
            color='tomato', label='Diabetes (1)',
            bw_adjust=0.8, ax=axes[2])
axes[2].axvline(x=0.5, color='black', linestyle='--',
                linewidth=1.2, label='Default (0.5)')
axes[2].axvline(x=best_thresh, color='gray', linestyle=':',
                linewidth=1.2, label=f'CV Threshold ({best_thresh:.2f})')
axes[2].set_xlabel('Predicted Probability of Diabetes')
axes[2].set_ylabel('Density')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('XGBoost Final Evaluation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()